# 11 · Agreement and directional disagreements

Operational view: where v1 and v2 vote the same, and where they flip.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from cross_model_drift.notebook import setup_model_session

nb = setup_model_session()
preds = pd.read_parquet(nb.artifacts / "reports" / "holdout_predictions.parquet")
n = len(preds)
summary = pd.Series(
    {
        "agreement": (preds["v1_pred"] == preds["v2_pred"]).mean(),
        "v1_neg_v2_pos": ((preds["v1_pred"] == 0) & (preds["v2_pred"] == 1)).mean(),
        "v1_pos_v2_neg": ((preds["v1_pred"] == 1) & (preds["v2_pred"] == 0)).mean(),
        "n": n,
    }
)
summary

In [ ]:
confusion = pd.crosstab(preds["v1_pred"], preds["v2_pred"], rownames=["v1"], colnames=["v2"])
fig, ax = plt.subplots(figsize=(4.8, 4.2))
sns.heatmap(confusion, annot=True, fmt=",d", cmap="Blues", ax=ax)
ax.set_title("v1 vs v2 predicted class")
nb.show(fig)
confusion

In [ ]:
flips = preds.loc[preds["v1_pred"] != preds["v2_pred"]].copy()
flips["direction"] = np.where(flips["v2_pred"] == 1, "v1 NEG → v2 POS", "v1 POS → v2 NEG")
flips.groupby("direction")["y_true"].agg(["count", "mean"]).rename(columns={"mean": "fraud_rate"})